# 03 - Creacion del target `caida_critica`

Objetivo: crear formalmente la variable objetivo del proyecto de clasificacion binaria.

Contexto: el EDA recomendo construir el target usando una variable de `produccion_principal` por pozo: `prod_pet` para pozos petroliferos y `prod_gas` para pozos gasiferos. El umbral inicial recomendado para explorar `caida_critica` es una caida futura mayor o igual al 30%.

Reglas de esta etapa:
- No se entrenan modelos.
- No se crean lags, medias moviles ni features avanzadas.
- Las variables que miran el mes siguiente se usan solo para construir el target.
- No se usan variables futuras como features.
- Se dejan fuera de `df_target` los registros donde no puede calcularse el target.

## 1. Importacion de librerias y configuracion

Se importan librerias basicas para procesamiento tabular y manejo de paths. No se importan librerias de modelado porque en este notebook no se entrenan modelos.

In [36]:
from pathlib import Path
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

TARGET_THRESHOLD_PCT = 30

## 2. Carga del dataset base

Se carga la base procesada generada en el notebook de calidad de datos. Esta base mantiene la unidad de analisis pozo-mes.

In [37]:
DATA_PATH = Path("../data/processed/produccion_no_convencional_base.csv")

df_base = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=["fecha_data", "fechaingreso"])
df_base["fecha_data"] = pd.to_datetime(df_base["fecha_data"], errors="coerce")

df_fe = df_base.copy()

print(f"Filas: {df_fe.shape[0]:,}")
print(f"Columnas: {df_fe.shape[1]:,}")
df_fe.head()

Filas: 405,996
Columnas: 40


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,YSUR,2016,1,135204,0.00,59.94,28.35,0.00,0.00,0.00,0.00,30.81,NaN,Plunger Lift,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-13(d),PREC,"3,500.00",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.79,-38.97,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31
1,YSUR,2018,1,155584,80.81,786.90,0.00,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2018-02-10 08:37:14.717426,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-262(d),LAJA,"3,847.00",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.80,-39.03,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
2,YSUR,2015,1,135203,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,Sin Sistema de Extracción,Parado Transitoriamente,Gasífero,NaN,2015-02-26 13:35:35.533458,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.ACO-12(d),PREC,"3,617.00",precuyo,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,NEUQUINA,Neuquén,-69.81,-38.96,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
3,YSUR,2017,1,155582,58.28,608.01,13.63,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2017-02-16 13:45:37.233373,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,YSR.RN.EFO-245(d),LAJA,"3,805.00",lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.85,-39.01,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
4,YSUR,2016,1,136137,0.00,432.17,45.52,0.00,0.00,0.00,0.00,31.00,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.Nq.Gu-1199(d),PREC,"3,375.00",precuyo,NDD,AL NORTE DE LA DORSAL,GUA,GUANACO,NEUQUINA,Neuquén,-69.22,-38.87,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31


## 3. Revision de valores negativos

Antes de crear el target, se identifican valores negativos en variables productivas. No se eliminan automaticamente. Para construir el target, los registros donde la produccion principal actual sea negativa quedan como no validos porque no permiten una interpretacion productiva confiable.

In [38]:
production_cols = ["prod_pet", "prod_gas", "prod_agua"]

negative_summary = pd.DataFrame(
    {
        "variable": production_cols,
        "casos_negativos": [(df_fe[col] < 0).sum() for col in production_cols],
        "valor_minimo": [df_fe[col].min() for col in production_cols],
    }
)

negative_summary

,variable,casos_negativos,valor_minimo
0,prod_pet,1,-0.00
1,prod_gas,2,-12.27
2,prod_agua,0,0.00


In [39]:
negative_mask = (df_fe[production_cols] < 0).any(axis=1)
negative_display_cols = [
    "idpozo", "anio", "mes", "fecha_data", "empresa", "provincia", "cuenca",
    "tipopozo", "tipoestado", "tipoextraccion", "prod_pet", "prod_gas", "prod_agua",
]
negative_records = df_fe.loc[negative_mask, negative_display_cols]
negative_records

,idpozo,anio,mes,fecha_data,empresa,provincia,cuenca,tipopozo,tipoestado,tipoextraccion,prod_pet,prod_gas,prod_agua
289972,153227,2020,5,2020-05-31,PLUSPETROL S.A.,Neuquén,NEUQUINA,Gasífero,Extracción Efectiva,Surgencia Natural,1.02,-7.52,0.00
290026,153228,2020,5,2020-05-31,PLUSPETROL S.A.,Neuquén,NEUQUINA,Gasífero,Extracción Efectiva,Surgencia Natural,1.67,-12.27,0.00
312211,73804,2008,12,2008-12-31,PETROBRAS ARGENTINA S.A.,Santa Cruz,AUSTRAL,Gasífero,En Estudio,Sin Sistema de Extracción,-0.00,0.00,0.00


## 4. Creacion de `produccion_principal`

Se define una variable productiva principal segun `tipopozo`:
- si el pozo es petrolifero, se usa `prod_pet`;
- si el pozo es gasifero, se usa `prod_gas`;
- para otros tipos de pozo, se deja `NaN`.

Se normaliza el texto de `tipopozo` para reconocer variantes con y sin tilde: `Petrolífero`, `Petrolifero`, `Gasífero`, `Gasifero`.

In [40]:
def normalize_text_for_matching(value: object) -> str:
    if pd.isna(value):
        return ""
    normalized = unicodedata.normalize("NFKD", str(value))
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    return ascii_text.strip().lower()


tipopozo_normalizado = df_fe["tipopozo"].map(normalize_text_for_matching)

petrolifero_mask = tipopozo_normalizado.eq("petrolifero")
gasifero_mask = tipopozo_normalizado.eq("gasifero")

df_fe["produccion_principal"] = np.nan
df_fe["criterio_produccion_principal"] = "fuera_definicion"

df_fe.loc[petrolifero_mask, "produccion_principal"] = df_fe.loc[petrolifero_mask, "prod_pet"]
df_fe.loc[petrolifero_mask, "criterio_produccion_principal"] = "prod_pet_por_pozo_petrolifero"

df_fe.loc[gasifero_mask, "produccion_principal"] = df_fe.loc[gasifero_mask, "prod_gas"]
df_fe.loc[gasifero_mask, "criterio_produccion_principal"] = "prod_gas_por_pozo_gasifero"

df_fe[["tipopozo", "prod_pet", "prod_gas", "produccion_principal", "criterio_produccion_principal"]].head()

,tipopozo,prod_pet,prod_gas,produccion_principal,criterio_produccion_principal
0,Gasífero,0.00,59.94,59.94,prod_gas_por_pozo_gasifero
1,Gasífero,80.81,786.90,786.90,prod_gas_por_pozo_gasifero
2,Gasífero,0.00,0.00,0.00,prod_gas_por_pozo_gasifero
3,Gasífero,58.28,608.01,608.01,prod_gas_por_pozo_gasifero
4,Gasífero,0.00,432.17,432.17,prod_gas_por_pozo_gasifero


## 5. Cobertura de la definicion de produccion principal

Se reporta cuanta informacion queda cubierta por la definicion y cuantos registros quedan fuera por no ser pozos petroliferos ni gasiferos.

In [41]:
registros_petroliferos = int(petrolifero_mask.sum())
registros_gasiferos = int(gasifero_mask.sum())
registros_fuera_definicion = int((~(petrolifero_mask | gasifero_mask)).sum())
pct_principal_nula = df_fe["produccion_principal"].isna().mean() * 100
registros_principal_no_positiva = int((df_fe["produccion_principal"] <= 0).sum())
registros_principal_negativa = int((df_fe["produccion_principal"] < 0).sum())

principal_definition_summary = pd.DataFrame(
    {
        "metrica": [
            "registros_petroliferos",
            "registros_gasiferos",
            "registros_fuera_definicion",
            "pct_produccion_principal_nula",
            "registros_produccion_principal_menor_o_igual_cero",
            "registros_produccion_principal_negativa",
        ],
        "valor": [
            registros_petroliferos,
            registros_gasiferos,
            registros_fuera_definicion,
            pct_principal_nula,
            registros_principal_no_positiva,
            registros_principal_negativa,
        ],
    }
)

principal_definition_summary

,metrica,valor
0,registros_petroliferos,"156,071.00"
1,registros_gasiferos,"221,257.00"
2,registros_fuera_definicion,"28,668.00"
3,pct_produccion_principal_nula,7.06
4,registros_produccion_principal_menor_o_igual_cero,"53,434.00"
5,registros_produccion_principal_negativa,2.00


In [42]:
principal_by_tipopozo = (
    df_fe.groupby(["tipopozo", "criterio_produccion_principal"], dropna=False)
    .agg(
        registros=("idpozo", "size"),
        pozos_unicos=("idpozo", "nunique"),
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

principal_by_tipopozo

,tipopozo,criterio_produccion_principal,registros,pozos_unicos
0,Gasífero,prod_gas_por_pozo_gasifero,221257,2448
4,Petrolífero,prod_pet_por_pozo_petrolifero,156071,2565
3,Otro tipo,fuera_definicion,27320,868
5,Sumidero,fuera_definicion,644,24
6,NaN,fuera_definicion,605,192
1,Inyección de Agua,fuera_definicion,56,4
2,Inyección de Gas,fuera_definicion,43,7


## 6. Produccion del mes siguiente y variacion futura

Se ordena el dataset por `idpozo` y `fecha_data`. Luego se calcula `produccion_principal_next` con `shift(-1)` dentro de cada pozo.

Estas variables miran el mes siguiente. Por lo tanto, solo se usan para construir `caida_critica` y no deben usarse como features en modelado.

In [43]:
df_fe = df_fe.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_fe["produccion_principal_next"] = df_fe.groupby("idpozo")["produccion_principal"].shift(-1)
df_fe["var_futura_principal_pct"] = np.nan

variation_valid_mask = (
    (df_fe["produccion_principal"] > 0)
    & df_fe["produccion_principal_next"].notna()
)

df_fe.loc[variation_valid_mask, "var_futura_principal_pct"] = (
    (
        df_fe.loc[variation_valid_mask, "produccion_principal_next"]
        - df_fe.loc[variation_valid_mask, "produccion_principal"]
    )
    / df_fe.loc[variation_valid_mask, "produccion_principal"]
    * 100
)

df_fe[["idpozo", "fecha_data", "produccion_principal", "produccion_principal_next", "var_futura_principal_pct"]].head(10)

,idpozo,fecha_data,produccion_principal,produccion_principal_next,var_futura_principal_pct
0,3640,2006-02-28,0.00,30.81,NaN
1,3640,2006-03-31,30.81,76.61,148.67
2,3640,2006-04-30,76.61,63.27,-17.41
3,3640,2006-05-31,63.27,55.04,-13.01
4,3640,2006-06-30,55.04,58.86,6.94
5,3640,2006-07-31,58.86,33.24,-43.53
6,3640,2006-08-31,33.24,37.29,12.17
7,3640,2006-09-30,37.29,64.17,72.08
8,3640,2006-10-31,64.17,62.86,-2.04
9,3640,2006-11-30,62.86,66.64,6.02


## 7. Creacion de `caida_critica`

Se define `caida_critica` con el umbral exploratorio recomendado en EDA:
- `caida_critica = 1` si `var_futura_principal_pct <= -30`;
- `caida_critica = 0` si `var_futura_principal_pct > -30`.

Los registros donde no se puede calcular `var_futura_principal_pct` quedan como no modelables en esta version.

In [44]:
df_fe["caida_critica"] = pd.Series(pd.NA, index=df_fe.index, dtype="Int64")

target_calculable_mask = df_fe["var_futura_principal_pct"].notna()

df_fe.loc[target_calculable_mask, "caida_critica"] = (
    df_fe.loc[target_calculable_mask, "var_futura_principal_pct"] <= -TARGET_THRESHOLD_PCT
).astype(int)

df_fe.loc[target_calculable_mask, [
    "idpozo", "fecha_data", "produccion_principal", "produccion_principal_next",
    "var_futura_principal_pct", "caida_critica"
]].head(10)

,idpozo,fecha_data,produccion_principal,produccion_principal_next,var_futura_principal_pct,caida_critica
1,3640,2006-03-31,30.81,76.61,148.67,0
2,3640,2006-04-30,76.61,63.27,-17.41,0
3,3640,2006-05-31,63.27,55.04,-13.01,0
4,3640,2006-06-30,55.04,58.86,6.94,0
5,3640,2006-07-31,58.86,33.24,-43.53,1
6,3640,2006-08-31,33.24,37.29,12.17,0
7,3640,2006-09-30,37.29,64.17,72.08,0
8,3640,2006-10-31,64.17,62.86,-2.04,0
9,3640,2006-11-30,62.86,66.64,6.02,0
10,3640,2006-12-31,66.64,51.73,-22.38,0


## 8. Registros modelables y no modelables

`df_target` contiene solo registros donde el target pudo calcularse. Para reducir riesgo de data leakage en etapas posteriores, no se guardan en `df_target` las columnas que miran el mes siguiente: `produccion_principal_next` y `var_futura_principal_pct`.

In [45]:
not_modelable_summary = pd.DataFrame(
    {
        "motivo": [
            "produccion_principal_nula_fuera_definicion",
            "produccion_principal_menor_o_igual_cero",
            "sin_mes_siguiente_observable",
            "total_sin_target_calculable",
        ],
        "registros": [
            int(df_fe["produccion_principal"].isna().sum()),
            int((df_fe["produccion_principal"].notna() & (df_fe["produccion_principal"] <= 0)).sum()),
            int(((df_fe["produccion_principal"] > 0) & df_fe["produccion_principal_next"].isna()).sum()),
            int((~target_calculable_mask).sum()),
        ],
    }
)

not_modelable_summary

,motivo,registros
0,produccion_principal_nula_fuera_definicion,28668
1,produccion_principal_menor_o_igual_cero,53434
2,sin_mes_siguiente_observable,4173
3,total_sin_target_calculable,86275


In [46]:
future_target_columns = ["produccion_principal_next", "var_futura_principal_pct"]

df_target = df_fe.loc[target_calculable_mask].copy()
df_target["caida_critica"] = df_target["caida_critica"].astype(int)
df_target = df_target.drop(columns=future_target_columns)

print(f"Dimensiones df_fe: {df_fe.shape[0]:,} filas x {df_fe.shape[1]:,} columnas")
print(f"Dimensiones df_target: {df_target.shape[0]:,} filas x {df_target.shape[1]:,} columnas")

Dimensiones df_fe: 405,996 filas x 45 columnas
Dimensiones df_target: 319,721 filas x 43 columnas


## 9. Resumen final del target

Se reportan dimensiones, registros eliminados por falta de target, distribucion de clases, rango temporal y cantidad de pozos unicos en la base modelable.

In [47]:
target_distribution = pd.DataFrame(
    {
        "cantidad": df_target["caida_critica"].value_counts().sort_index(),
        "porcentaje": df_target["caida_critica"].value_counts(normalize=True).sort_index() * 100,
    }
)
target_distribution.index.name = "caida_critica"

target_distribution

,cantidad,porcentaje
caida_critica,,
0,281911,88.17
1,37810,11.83


In [48]:
target_final_summary = pd.DataFrame(
    {
        "metrica": [
            "filas_df_fe",
            "columnas_df_fe",
            "filas_df_target",
            "columnas_df_target",
            "registros_eliminados_por_falta_de_target",
            "fecha_min_df_target",
            "fecha_max_df_target",
            "pozos_unicos_df_target",
        ],
        "valor": [
            df_fe.shape[0],
            df_fe.shape[1],
            df_target.shape[0],
            df_target.shape[1],
            df_fe.shape[0] - df_target.shape[0],
            df_target["fecha_data"].min(),
            df_target["fecha_data"].max(),
            df_target["idpozo"].nunique(),
        ],
    }
)

target_final_summary

,metrica,valor
0,filas_df_fe,405996
1,columnas_df_fe,45
2,filas_df_target,319721
3,columnas_df_target,43
4,registros_eliminados_por_falta_de_target,86275
5,fecha_min_df_target,2006-01-31 00:00:00
6,fecha_max_df_target,2026-03-31 00:00:00
7,pozos_unicos_df_target,4681


## 10. Guardado de `df_target`

Se guarda la base modelable con el target `caida_critica`. Las columnas que miran el futuro se usaron para construir el target, pero no se guardan en `df_target` para evitar que luego sean usadas como features por error.

In [49]:
OUTPUT_PATH = Path("../data/processed/df_target_caida_critica.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_target.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        "archivo_guardado": [str(OUTPUT_PATH)],
        "filas": [df_target.shape[0]],
        "columnas": [df_target.shape[1]],
    }
)

,archivo_guardado,filas,columnas
0,..\data\processed\df_target_caida_critica.csv,319721,43


## Conclusiones sobre creacion del target

- **Construccion de `produccion_principal`:** se uso `prod_pet` para registros de pozos petroliferos y `prod_gas` para registros de pozos gasiferos, segun `tipopozo`. Las variantes con y sin tilde fueron contempladas mediante normalizacion de texto. Otros tipos de pozo quedaron con `produccion_principal = NaN`.
- **Construccion de `caida_critica`:** se calculo la produccion principal del mes siguiente por `idpozo`, ordenando por `fecha_data`. Luego se calculo la variacion futura porcentual y se definio `caida_critica = 1` cuando la caida fue mayor o igual al 30%, es decir, `var_futura_principal_pct <= -30`. Si la variacion fue mayor a `-30`, se definio `caida_critica = 0`.
- **Registros fuera de `df_target`:** quedaron fuera los registros sin produccion principal definida, los registros con produccion principal actual menor o igual a cero y los registros sin mes siguiente observable para el mismo pozo. Tambien quedan fuera, para el calculo del target, los registros donde la produccion principal actual es negativa.
- **Balance final del target:** la clase `0` representa aproximadamente el 88,17% de los registros modelables y la clase `1` aproximadamente el 11,83%. Hay desbalance, pero no es extremo para una primera version de clasificacion binaria.
- **Variables futuras que no deben usarse como features:** `produccion_principal_next` y `var_futura_principal_pct` miran el mes siguiente. Se usaron solo para construir `caida_critica` y no se guardaron en `df_target`.
- **Cuidados pendientes para feature engineering:** las proximas features deben construirse usando solo informacion disponible hasta el mes actual. Lags, medias moviles y tendencias deberan calcularse por `idpozo`, respetando el orden temporal y evitando cualquier agregacion que incluya meses futuros.
- **Prevencion de data leakage:** el target mira el mes siguiente, pero las features no pueden mirar el mes siguiente. En modelado tambien sera necesario usar un split temporal para simular el escenario real de prediccion.

## Validacion de continuidad mensual

El target `caida_critica` fue creado con `shift(-1)` por `idpozo`. Esta validacion revisa si el siguiente registro observado para cada pozo corresponde realmente al mes calendario siguiente.

No se modifica la definicion actual del target en esta seccion. El objetivo es diagnosticar continuidad temporal y documentar una decision metodologica pendiente antes de avanzar con feature engineering y modelado.

### Calculo del siguiente registro observado

Se ordena por `idpozo` y `fecha_data`, y se crea `fecha_siguiente_observada` con `shift(-1)` dentro de cada pozo.

In [50]:
df_continuidad = df_fe.copy()
df_continuidad["fecha_data"] = pd.to_datetime(df_continuidad["fecha_data"], errors="coerce")
df_continuidad = df_continuidad.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_continuidad["fecha_siguiente_observada"] = df_continuidad.groupby("idpozo")["fecha_data"].shift(-1)

mes_actual_idx = df_continuidad["fecha_data"].dt.year * 12 + df_continuidad["fecha_data"].dt.month
mes_siguiente_idx = (
    df_continuidad["fecha_siguiente_observada"].dt.year * 12
    + df_continuidad["fecha_siguiente_observada"].dt.month
)

df_continuidad["gap_meses_hasta_siguiente"] = mes_siguiente_idx - mes_actual_idx

df_continuidad[["idpozo", "fecha_data", "fecha_siguiente_observada", "gap_meses_hasta_siguiente"]].head(10)

,idpozo,fecha_data,fecha_siguiente_observada,gap_meses_hasta_siguiente
0,3640,2006-02-28,2006-03-31,1.00
1,3640,2006-03-31,2006-04-30,1.00
2,3640,2006-04-30,2006-05-31,1.00
3,3640,2006-05-31,2006-06-30,1.00
4,3640,2006-06-30,2006-07-31,1.00
5,3640,2006-07-31,2006-08-31,1.00
6,3640,2006-08-31,2006-09-30,1.00
7,3640,2006-09-30,2006-10-31,1.00
8,3640,2006-10-31,2006-11-30,1.00
9,3640,2006-11-30,2006-12-31,1.00


### Resumen de continuidad mensual

Se calculan pares con siguiente registro observado. Los ultimos registros de cada pozo quedan sin `fecha_siguiente_observada` y no forman parte del denominador de continuidad.

In [51]:
gap_valid_mask = df_continuidad["gap_meses_hasta_siguiente"].notna()
total_pares_con_siguiente = int(gap_valid_mask.sum())

pares_gap_1 = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] == 1).sum())
pares_gap_mayor_1 = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] > 1).sum())
pares_gap_otro = int((df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"] <= 0).sum())
registros_sin_siguiente = int(df_continuidad["fecha_siguiente_observada"].isna().sum())

continuidad_summary = pd.DataFrame(
    {
        "categoria": [
            "gap_1_mes",
            "gap_mayor_a_1_mes",
            "gap_menor_o_igual_a_0",
            "sin_siguiente_observado",
        ],
        "cantidad": [
            pares_gap_1,
            pares_gap_mayor_1,
            pares_gap_otro,
            registros_sin_siguiente,
        ],
        "porcentaje_sobre_pares_con_siguiente": [
            pares_gap_1 / total_pares_con_siguiente * 100,
            pares_gap_mayor_1 / total_pares_con_siguiente * 100,
            pares_gap_otro / total_pares_con_siguiente * 100,
            np.nan,
        ],
    }
)

continuidad_summary

,categoria,cantidad,porcentaje_sobre_pares_con_siguiente
0,gap_1_mes,400236,99.79
1,gap_mayor_a_1_mes,831,0.21
2,gap_menor_o_igual_a_0,0,0.00
3,sin_siguiente_observado,4929,NaN


In [52]:
gap_distribution = (
    df_continuidad.loc[gap_valid_mask, "gap_meses_hasta_siguiente"]
    .value_counts()
    .sort_index()
    .rename_axis("gap_meses_hasta_siguiente")
    .reset_index(name="cantidad")
)

gap_distribution.head(20)

,gap_meses_hasta_siguiente,cantidad
0,1.00,400236
1,2.00,610
2,3.00,44
3,4.00,13
4,5.00,7
5,6.00,5
6,7.00,8
7,8.00,5
8,9.00,3
9,10.00,1


### Cruces de gaps mayores a 1 mes

Se analizan los casos donde el siguiente registro observado no corresponde al mes calendario siguiente. Esto ayuda a detectar si los gaps se concentran en ciertos periodos, estados, tipos de pozo o empresas.

In [53]:
gaps_mayores_1 = df_continuidad.loc[df_continuidad["gap_meses_hasta_siguiente"] > 1].copy()
gaps_mayores_1["anio_gap"] = gaps_mayores_1["fecha_data"].dt.year

gaps_por_anio = (
    gaps_mayores_1.groupby("anio_gap")
    .size()
    .rename("cantidad_gaps_mayores_1")
    .reset_index()
    .sort_values("cantidad_gaps_mayores_1", ascending=False)
)

gaps_por_anio.head(20)

,anio_gap,cantidad_gaps_mayores_1
7,2014,218
15,2024,164
10,2018,156
1,2008,136
11,2019,31
16,2025,30
8,2015,18
14,2023,18
5,2012,16
6,2013,13


In [54]:
gaps_por_tipoestado = (
    gaps_mayores_1["tipoestado"]
    .value_counts(dropna=False)
    .rename_axis("tipoestado")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_tipoestado.head(15)

,tipoestado,cantidad_gaps_mayores_1
0,Extracción Efectiva,295
1,Abandonado,226
2,En Estudio,140
3,Parado Transitoriamente,73
4,Otras Situación Inactivo,41
5,En Reserva de Gas,38
6,NaN,7
7,A Abandonar,6
8,Abandono Temporario,2
9,En Reserva para Recup. Sec./Asist.,1


In [55]:
gaps_por_tipopozo = (
    gaps_mayores_1["tipopozo"]
    .value_counts(dropna=False)
    .rename_axis("tipopozo")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_tipopozo.head(15)

,tipopozo,cantidad_gaps_mayores_1
0,Otro tipo,391
1,Gasífero,230
2,Petrolífero,202
3,NaN,7
4,Sumidero,1


In [56]:
gaps_por_empresa = (
    gaps_mayores_1["empresa"]
    .value_counts(dropna=False)
    .rename_axis("empresa")
    .reset_index(name="cantidad_gaps_mayores_1")
)

gaps_por_empresa.head(15)

,empresa,cantidad_gaps_mayores_1
0,YPF S.A.,345
1,PLUSPETROL S.A.,163
2,SHELL ARGENTINA S.A.,161
3,TOTAL AUSTRAL S.A.,39
4,QUINTANA E&P ARGENTINA S.R.L.,28
5,O&G DEVELOPMENTS LTD S.A.,28
6,TECPETROL S.A.,17
7,APACHE ENERGIA ARGENTINA S.R.L.,11
8,PETROLERA EL TREBOL S.A.,11
9,GAS Y PETROLEO DEL NEUQUEN S.A.,7


### Conclusion sobre continuidad mensual

La gran mayoria de los pares consecutivos por pozo tiene continuidad mensual: 400.236 pares presentan gap de 1 mes, equivalentes al 99,79% de los pares con siguiente registro observado. Solo 831 pares tienen gaps mayores a 1 mes, equivalentes al 0,21%.

Para una primera version exploratoria del proyecto, usar el siguiente registro observado como proxy del mes siguiente es casi equivalente a usar el mes calendario siguiente, porque los gaps son muy poco frecuentes. Sin embargo, desde el punto de vista metodologico, `caida_critica` se define como una caida en el mes siguiente. Por lo tanto, conviene exigir `gap_meses_hasta_siguiente == 1` antes de cerrar la base de modelado.

Decision recomendada para el proyecto del curso: documentar esta validacion y dejar como decision pendiente ajustar el target para que solo se calcule cuando el siguiente registro observado corresponda exactamente al mes calendario siguiente. El impacto esperado en cantidad de datos es bajo, pero mejora la consistencia temporal del target y reduce riesgo de interpretar saltos de varios meses como caidas mensuales.

## Target final con continuidad mensual estricta

A partir de la validacion anterior, se ajusta la construccion formal del target para exigir continuidad mensual estricta. Esto significa que `caida_critica` solo se calcula cuando el siguiente registro observado del mismo pozo corresponde exactamente al mes calendario siguiente (`gap_meses_hasta_siguiente == 1`).

Este ajuste mejora la consistencia temporal del proyecto: evita interpretar saltos de varios meses como si fueran una caida mensual y reduce el riesgo de leakage temporal o de una definicion ambigua del evento objetivo.

### Recalculo estricto del target

Se conserva `df_target` como version previa para comparar. Luego se recalcula el target sobre una copia de `df_fe`, ordenando por `idpozo` y `fecha_data`, y exigiendo que el siguiente registro observado este a un mes calendario de distancia.

In [57]:
df_target_previo = df_target.copy() if "df_target" in globals() else None

df_fe_strict = df_fe.copy()
df_fe_strict["fecha_data"] = pd.to_datetime(df_fe_strict["fecha_data"], errors="coerce")
df_fe_strict = df_fe_strict.sort_values(["idpozo", "fecha_data"]).reset_index(drop=True)

df_fe_strict["fecha_siguiente_observada"] = df_fe_strict.groupby("idpozo")["fecha_data"].shift(-1)

mes_actual_idx = df_fe_strict["fecha_data"].dt.year * 12 + df_fe_strict["fecha_data"].dt.month
mes_siguiente_idx = (
    df_fe_strict["fecha_siguiente_observada"].dt.year * 12
    + df_fe_strict["fecha_siguiente_observada"].dt.month
)

df_fe_strict["gap_meses_hasta_siguiente"] = mes_siguiente_idx - mes_actual_idx
df_fe_strict["produccion_principal_next"] = df_fe_strict.groupby("idpozo")["produccion_principal"].shift(-1)
df_fe_strict["var_futura_principal_pct"] = np.nan

strict_target_mask = (
    (df_fe_strict["produccion_principal"] > 0)
    & df_fe_strict["produccion_principal_next"].notna()
    & (df_fe_strict["gap_meses_hasta_siguiente"] == 1)
)

df_fe_strict.loc[strict_target_mask, "var_futura_principal_pct"] = (
    (
        df_fe_strict.loc[strict_target_mask, "produccion_principal_next"]
        - df_fe_strict.loc[strict_target_mask, "produccion_principal"]
    )
    / df_fe_strict.loc[strict_target_mask, "produccion_principal"]
    * 100
)

df_fe_strict["caida_critica"] = pd.Series(pd.NA, index=df_fe_strict.index, dtype="Int64")
df_fe_strict.loc[strict_target_mask, "caida_critica"] = (
    df_fe_strict.loc[strict_target_mask, "var_futura_principal_pct"] <= -TARGET_THRESHOLD_PCT
).astype(int)

df_fe_strict.loc[strict_target_mask, [
    "idpozo", "fecha_data", "fecha_siguiente_observada", "gap_meses_hasta_siguiente",
    "produccion_principal", "produccion_principal_next", "var_futura_principal_pct", "caida_critica"
]].head(10)

,idpozo,fecha_data,fecha_siguiente_observada,gap_meses_hasta_siguiente,produccion_principal,produccion_principal_next,var_futura_principal_pct,caida_critica
1,3640,2006-03-31,2006-04-30,1.00,30.81,76.61,148.67,0
2,3640,2006-04-30,2006-05-31,1.00,76.61,63.27,-17.41,0
3,3640,2006-05-31,2006-06-30,1.00,63.27,55.04,-13.01,0
4,3640,2006-06-30,2006-07-31,1.00,55.04,58.86,6.94,0
5,3640,2006-07-31,2006-08-31,1.00,58.86,33.24,-43.53,1
6,3640,2006-08-31,2006-09-30,1.00,33.24,37.29,12.17,0
7,3640,2006-09-30,2006-10-31,1.00,37.29,64.17,72.08,0
8,3640,2006-10-31,2006-11-30,1.00,64.17,62.86,-2.04,0
9,3640,2006-11-30,2006-12-31,1.00,62.86,66.64,6.02,0
10,3640,2006-12-31,2007-01-31,1.00,66.64,51.73,-22.38,0


### Base final modelable

Se crea `df_target_final` con registros donde el target estricto pudo calcularse. Las columnas que miran el futuro se excluyen del dataset final para que no puedan usarse como features por error.

In [58]:
future_columns_strict = [
    "produccion_principal_next",
    "fecha_siguiente_observada",
    "gap_meses_hasta_siguiente",
    "var_futura_principal_pct",
]

df_target_final = df_fe_strict.loc[strict_target_mask].copy()
df_target_final["caida_critica"] = df_target_final["caida_critica"].astype(int)
df_target_final = df_target_final.drop(columns=[col for col in future_columns_strict if col in df_target_final.columns])

print(f"Dimensiones df_target_final: {df_target_final.shape[0]:,} filas x {df_target_final.shape[1]:,} columnas")

Dimensiones df_target_final: 319,453 filas x 43 columnas


### Comparacion antes y despues del ajuste

Se compara la version previa del target, basada en el siguiente registro observado, contra la version final estricta que exige continuidad mensual exacta.

In [59]:
filas_target_previo = len(df_target_previo) if df_target_previo is not None else np.nan
filas_target_final = len(df_target_final)
registros_eliminados_por_continuidad = filas_target_previo - filas_target_final

resumen_comparativo_target = pd.DataFrame(
    {
        "metrica": [
            "filas_df_target_previo",
            "filas_df_target_final",
            "registros_eliminados_por_exigir_continuidad_mensual",
            "pct_eliminado_sobre_target_previo",
            "fecha_min_df_target_final",
            "fecha_max_df_target_final",
            "pozos_unicos_df_target_final",
        ],
        "valor": [
            filas_target_previo,
            filas_target_final,
            registros_eliminados_por_continuidad,
            registros_eliminados_por_continuidad / filas_target_previo * 100,
            df_target_final["fecha_data"].min(),
            df_target_final["fecha_data"].max(),
            df_target_final["idpozo"].nunique(),
        ],
    }
)

resumen_comparativo_target

,metrica,valor
0,filas_df_target_previo,319721
1,filas_df_target_final,319453
2,registros_eliminados_por_exigir_continuidad_me...,268
3,pct_eliminado_sobre_target_previo,0.08
4,fecha_min_df_target_final,2006-01-31 00:00:00
5,fecha_max_df_target_final,2026-03-31 00:00:00
6,pozos_unicos_df_target_final,4681


In [60]:
def summarize_target_distribution(df_input: pd.DataFrame, version: str) -> pd.DataFrame:
    counts = df_input["caida_critica"].value_counts().sort_index()
    pct = df_input["caida_critica"].value_counts(normalize=True).sort_index() * 100
    return pd.DataFrame(
        {
            "version": version,
            "caida_critica": counts.index.astype(int),
            "cantidad": counts.values,
            "porcentaje": pct.values,
        }
    )


target_distribution_comparison = pd.concat(
    [
        summarize_target_distribution(df_target_previo, "previo_shift_sin_gap_estricto"),
        summarize_target_distribution(df_target_final, "final_gap_1_mes"),
    ],
    ignore_index=True,
)

target_distribution_comparison

,version,caida_critica,cantidad,porcentaje
0,previo_shift_sin_gap_estricto,0,281911,88.17
1,previo_shift_sin_gap_estricto,1,37810,11.83
2,final_gap_1_mes,0,281723,88.19
3,final_gap_1_mes,1,37730,11.81


### Distribucion final de `caida_critica`

Resumen absoluto y porcentual de la clase objetivo final con continuidad mensual estricta.

In [61]:
target_final_distribution = pd.DataFrame(
    {
        "cantidad": df_target_final["caida_critica"].value_counts().sort_index(),
        "porcentaje": df_target_final["caida_critica"].value_counts(normalize=True).sort_index() * 100,
    }
)
target_final_distribution.index.name = "caida_critica"

porcentaje_clase_positiva_final = float(
    target_final_distribution.loc[1, "porcentaje"] if 1 in target_final_distribution.index else 0
)

target_final_distribution

,cantidad,porcentaje
caida_critica,,
0,281723,88.19
1,37730,11.81


In [62]:
target_final_metrics = pd.DataFrame(
    {
        "metrica": [
            "filas_df_target_final",
            "columnas_df_target_final",
            "fecha_min_df_target_final",
            "fecha_max_df_target_final",
            "pozos_unicos_df_target_final",
            "porcentaje_clase_positiva_final",
        ],
        "valor": [
            df_target_final.shape[0],
            df_target_final.shape[1],
            df_target_final["fecha_data"].min(),
            df_target_final["fecha_data"].max(),
            df_target_final["idpozo"].nunique(),
            porcentaje_clase_positiva_final,
        ],
    }
)

target_final_metrics

,metrica,valor
0,filas_df_target_final,319453
1,columnas_df_target_final,43
2,fecha_min_df_target_final,2006-01-31 00:00:00
3,fecha_max_df_target_final,2026-03-31 00:00:00
4,pozos_unicos_df_target_final,4681
5,porcentaje_clase_positiva_final,11.81


### Guardado del dataset final

Se sobrescribe `../data/processed/df_target_caida_critica.csv` con la version final modelable que exige continuidad mensual estricta.

In [63]:
OUTPUT_PATH = Path("../data/processed/df_target_caida_critica.csv")
df_target_final.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        "archivo_guardado": [str(OUTPUT_PATH)],
        "filas": [df_target_final.shape[0]],
        "columnas": [df_target_final.shape[1]],
    }
)

,archivo_guardado,filas,columnas
0,..\data\processed\df_target_caida_critica.csv,319453,43


### Conclusion del target final con continuidad mensual estricta

Se exige `gap_meses_hasta_siguiente == 1` porque el objetivo del proyecto es predecir una caida critica en el mes siguiente. Si se compararan registros separados por varios meses, el target podria representar una caida acumulada entre observaciones no consecutivas y no una caida mensual.

El ajuste elimina 268 registros respecto de la version previa del target, un impacto muy bajo frente al tamano total de la base modelable. El balance final queda practicamente estable: la clase positiva representa aproximadamente el 11,81% de los registros y la clase negativa aproximadamente el 88,19%.

`produccion_principal_next`, `fecha_siguiente_observada`, `gap_meses_hasta_siguiente` y `var_futura_principal_pct` no deben usarse como features del modelo porque contienen informacion del futuro o informacion usada directamente para construir el target. Estas variables solo sirven para definir `caida_critica` y validar su consistencia temporal.

Este ajuste mejora la consistencia temporal del proyecto, fortalece la prevencion de data leakage y deja una base final mas alineada con el enunciado del problema: clasificar si un pozo tendra una caida critica de produccion en el mes calendario siguiente.